## 📚 A Book Recommendation System: Navigating the Amazon Catalog

### 📖 Project Overview
This project builds a high-performance **Hybrid Recommendation Engine** using a massive dataset of Amazon Book Reviews. The goal is to move beyond simple "Top Rated" lists and create a system that truly understands both the **content** of a book and the **behavior** of the readers. By the end of this pipeline, the system suggests obscure gems that a user is statistically and conceptually likely to love.

Datasets taken from: https://www.kaggle.com/datasets/mohamedbakhet/amazon-books-reviews/data?select=Books_rating.csv

---

### 📊 The Data Foundation
The system leverages two primary datasets to fuel the engine:

1.  **Books Metadata**: A comprehensive catalog containing titles, descriptions, authors, and genres for over 200,000 unique books. This provides the "Context" for the recommendations.
2.  **User Ratings**: A collection of millions of individual reviews and scores. This provides the "Social Proof" and behavioral patterns needed to find similar tastes among users.

---

#### 📜 Three-Stage Strategy

####  Goal 1: Item-to-Item Semantic Recommendations
* **Objective:** Identify and provide similar books recommendation.
* **Recommendation Type:** Content-Based Filtering
* **Description:** This system utilizes **Natural Language Processing (NLP)** and **Deep Learning** to perform semantic search. Unlike simple keyword matching, this filter understands the context of the plot, the specific expertise of the authors, and the nuances of categorized genres.
* **How it works:** The system uses AI models (`all-MiniLM-L6-v2`) to turn text into 384-dimensional vectors (Embeddings). This allows the system to recognize that a book about "WWII Fighter Planes" is conceptually linked to "Aviation History," even if the words in the title are completely different.
* **Key Output:** `books_embeddings_matrix.npy` (209,452 books × 384 features)

####  Goal 2: User-to-User "Similar People" Recommendations
* **Objective:** Identify and suggest books that similar people like.
* **Recommendation Type:** Collaborative Filtering (Memory-Based)
* **Description:** This system uses a "wisdom of the crowd" approach. It calculates similarity between users based on their shared rating history. If User A and User B both gave 5 stars to the same 10 books, the system will recommend other high-rated books from User B’s library that User A has not yet discovered.
* **How it works:** Using Collaborative Filtering **(KNN)** with cosine similarity, the system finds "Taste Neighbors"—users who have rated the same books similarly to you. If your neighbors loved a book you haven't read yet, that book becomes a top candidate for you.
* **Key Parameters:** 30 neighbors | Cosine similarity | Mean-centered ratings | 3-level fallback (Personalized → Genre-based → Popular)

####  Goal 3: Personalized Taste-Based Recommendations (The Hybrid Intelligence Engine)
* **Objective:** Synthesize personal preference with community wisdom to find the "Perfect Match."
* **Recommendation Type:** Hybrid Synthesis (Content-Semantic + Behavioral Collaborative Filtering)
* **Description:** This is the "Master Brain" of the project. It moves beyond simple matching by identifying the specific **multidimensional intersection** of a user's taste—where collaborative signals (what similar people like) meet semantic meaning (what the book is actually about). The result is a curated list that feels both familiar and refreshingly personal.
* **How it works:** 
    1.  **User Profile Vector:** Averages embeddings of user's favorite books (≥4 stars) → 384-dim "taste DNA"
    2.  **Weighted Fusion:** `Final Score = (CF × 0.7) + (Semantic × 0.3)`
    3.  **Smart Filtering:** Aggressive deduplication (removes parentheses, conjunctions, punctuation) to strip out different editions/formats
    4.  **Final Ranking:** Curated Top 5 list ranked by hybrid confidence score
* **Key Parameters:** 100 neighbors | 70% CF / 30% Semantic | Aggressive fingerprint deduplication

### 🗂️ Project Index: Step-by-Step Architecture

Below is the roadmap of the project, organized by the execution order of the notebooks:

| # | Notebook | Scope | Key Tasks | Final Output |
|---|---|---|---|---|
| 01 | [01_env_verification.ipynb](./01_env_verification.ipynb) | Environment & Hardware Verification | Verification of Python 3.11+, high-performance data libraries (Polars, DuckDB, FAISS), and stress-testing the **Apple Silicon (M3 Pro) GPU** via MPS | Environment ready |
| 02 | [02_EDA_books_data.ipynb](./02_EDA_books_data.ipynb) | EDA - Metadata cleaning and catalog preparation | Processing 173 MB of raw book data, parsing complex nested strings for authors/genres, and optimizing storage by converting to **Apache Parquet** | `books_metadata_only.parquet` (209,452 books) |
| 03 | [03_Recsys_Book_Data.ipynb](./03_Recsys_Book_Data.ipynb) | Semantic Book Embeddings — Goal 1 Foundation | Leveraging **Sentence-Transformers** to generate 384-dimensional dense embeddings of book titles | `books_embeddings_matrix.npy` (209,452 × 384) |
| 04 | [04_EDA_Books_ratings.ipynb](./04_EDA_Books_ratings.ipynb) | EDA - Interaction data cleaning and behavioral audit | Processing 1.1GB of ratings. Cleaning millions of user reviews, filtering for high-confidence users (5+ reviews), and performing **Rating Centering** to remove user bias | `master_ratings_metadata.parquet` (385,727 ratings, 28,750 users, 18,335 books) |
| 05 | [05_Recsys_goal2.ipynb](./05_Recsys_goal2.ipynb) | Collaborative Filtering — Goal 2 | Implementing **Collaborative Filtering** using KNN with cosine similarity to find "Taste Neighbors" | Sparse matrix (28,750 × 18,335) + KNN model |
| 06 | [06_Recsys_goal3.ipynb](./06_Recsys_goal3.ipynb) | Hybrid Intelligence Engine — Goal 3 | Developing the final **Hybrid Ranker** merging semantic meanings (70% CF + 30% Semantic) with aggressive deduplication | Top-5 personalized recommendations per user |

### 🏆 Final Results & Key Achievements

| Goal | Success Metric | Result |
|------|----------------|--------|
| **Goal 1** | Embedding quality | 209,452 books → 384-dim vectors, 15x faster Parquet loading |
| **Goal 2** | KNN neighbor quality | 28,750 users × 18,335 books matrix, 30 neighbors per query |
| **Goal 3** | Recommendation relevance | Hybrid score (CF + Semantic) with 70/30 weighting, aggressive deduplication |

### Example Recommendation Output

```text
TOP 5 HYBRID RECOMMENDATIONS (Behavior + Meaning):
--------------------------------------------------------------------------------
#   | BOOK TITLE                                              | SCORE
--------------------------------------------------------------------------------
1   | Jonathan Strange & Mr. Norrell                           | 0.7478
    └─ Author: Susanna Clarke | Genre: Fiction
2   | Five People You Meet in Heaven                          | 0.7142
    └─ Author: Mitch Albom | Genre: Fiction
3   | Night Play (Dark-Hunter, Book 6)                        | 0.6774
    └─ Author: Sherrilyn Kenyon | Genre: Love stories
--------------------------------------------------------------------------------

### 🗺️ Project Data Flow Architecture

```text
┌─────────────────────────────────────────────────────────────────┐
│                    EDA / DATA PREPARATION                       │
│                       (Notebook 02 & 04)                        │
├─────────────────────────────────────────────────────────────────┤
│  • Clean metadata: remove duplicates, parse stringified lists   │
│  • Generate book embeddings using Sentence-Transformers         │
│  • Filter ratings: keep users & books with ≥5 ratings          │
│  • Apply iterative threshold filtering (remove circular deps)   │
│  • Add canonical_title fingerprint (typo correction)           │
│  • Save outputs:                                                │
│    - books_metadata_only.parquet (209,452 books)               │
│    - books_embeddings_matrix.npy (209,452 × 384)               │
│    - master_ratings_metadata.parquet (385,727 ratings)         │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────┐
│                    GOAL 1 — Semantic                            │
│                       (Notebook 03)                             │
├─────────────────────────────────────────────────────────────────┤
│  • Load cleaned metadata                                        │
│  • Generate 384-dim embeddings from book descriptions           │
│  • Save embedding matrix for similarity search                  │
│  • Output: books_embeddings_matrix.npy                         │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────┐
│                    GOAL 2 — Collaborative                       │
│                       (Notebook 05)                             │
├─────────────────────────────────────────────────────────────────┤
│  • Load filtered ratings (385,727 interactions)                │
│  • Mean-center ratings (remove user bias)                      │
│  • Build sparse matrix (28,750 users × 18,335 books)           │
│  • Train KNN with cosine similarity (30 neighbors)             │
│  • Implement 3-level fallback (Personalized → Genre → Popular) │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────┐
│                    GOAL 3 — Hybrid                              │
│                       (Notebook 06)                             │
├─────────────────────────────────────────────────────────────────┤
│  • Load: ratings + metadata + embeddings                       │
│  • Build user profile vector (average of favorite book vectors)│
│  • Generate CF candidates (100 neighbors)                      │
│  • Calculate semantic similarity (cosine with profile vector)  │
│  • Fuse scores: Hybrid = (CF × 0.7) + (Semantic × 0.3)         │
│  • Apply aggressive deduplication (fingerprint normalization)  │
│  • Return Top-5 personalized recommendations                   │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────────┐
│                      FINAL OUTPUT                               │
├─────────────────────────────────────────────────────────────────┤
│  • For any user with ≥5 ratings → Top-5 personalized books     │
│  • Hybrid recommendations (crowd wisdom + content matching)    │
│  • Clean, deduplicated, ranked by confidence score             │
└─────────────────────────────────────────────────────────────────┘

## References

### Netflix & OTT Platforms

1. Lankester, S., Slokom, M., Bertoli, G. C., Vizcaino, M., Beauxis-Aussalet, E., & Hollink, L. (2025). FedFlex: Federated Learning for Diverse Netflix Recommendations. *arXiv preprint*, arXiv:2507.21115. [citation:7]

2. Kashyap, P., Barman, B., & Kakati, C. (2025). Predictive Analysis of Netflix User Data Using a Hybrid Model of Naive Bayes Classifier and K-Means Clustering. In *Proceedings of ICSSSM 2025*. Atlantis Press. [citation:1]

3. Kim, S. (2025). The Impact of Personalized Recommendation Algorithms on User Experience and Satisfaction in OTT Services. *Industry Promotion Research*, 10(4), 285-297. [citation:4]

### Spotify

4. Feijer, D., Abdollahpouri, H., Gupta, S., Clare, A., Wen, Y., Wasson, T., Dimakopoulou, M., Nazari, Z., Kretschman, K., & Lalmas, M. (2025). Calibrated Recommendations with Contextual Bandits. *ACM RecSys '25, CONSEQUENCES workshop*. [citation:2]

5. Wu, M., & Lian, X. (2025). Content-Aware Music Recommendation via Neural Networks and Acoustic Feature Analysis. *Journal of Platform Technology*, 13(2), 3-14. [citation:5]

### Hybrid Recommendation Systems

6. Haw, S. C., Jayaram, J., & Anaam, E. (2025). A Hybrid Recommender Model for Health Supplements in E-commerce. *AIP Conference Proceedings*, 3367(1), 030009. [citation:3]